# Homework 12: Design of Experiments

This homework covers Lectures 38-39 (design of experiments). One dataset:
* `dataset_ceramic_doe` -- 32 sintered silicon-nitride specimens, a genuine complete, randomized
  2^5 factorial from NIST's Ceramics Division. You saw this dataset Friday; Problems 1-2 ask you
  to go further with it than lecture time allowed.

Problem 3 asks you to design a small factorial experiment of your own for a scenario with no
provided dataset -- there is no code to run and no "right" numeric answer, only a judged design.

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

*Only* the `ipynb` file type will be accepted.

Save a copy of the notebook right away to avoid losing your work!

# Problem 0 (0 pts): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work -- critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

## Data file for Problems 1-2

## Dataset: Ceramic Grinding Strength -- a real 2^5 factorial DOE

A complete, randomized 2^5 factorial experiment from NIST's Ceramics Division
(Said Jahanmir): 32 sintered reaction-bonded silicon nitride specimens ground
under every combination of 5 two-level factors (table speed, feed rate, wheel
grit, grinding direction, batch), with mean flexural strength as the response.
Coded `-1`/`+1` columns give the design matrix; matching physical-unit columns
(`table_speed_m_s`, `feed_rate_mm`, `wheel_grit`, `direction`, `batch`) give the
real levels. `run_order` is the actual randomized execution order (not the
design-matrix row order) -- correlation of strength with run_order is only
r=0.10, i.e. no meaningful time-drift confound, unlike a prior DOE candidate
this course tried and rejected for exactly that problem.

Source: NIST/SEMATECH e-Handbook of Statistical Methods, `CERAMIC.DAT`
(public domain, U.S. government work -- attribute as NIST/Said Jahanmir).
https://www.itl.nist.gov/div898/handbook/pri/section4/pri471.htm

In [1]:
import os
import pandas as pd

_file = 'ceramic_doe.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

# set up features (coded factor levels) and target (strength)
x = data[['table_speed_coded', 'feed_rate_coded', 'wheel_grit_coded', 'direction_coded', 'batch_coded']]
y = data['strength']

print(f'{len(data)} runs, {x.shape[1]} factors (2^{x.shape[1]} full factorial), '
      f'strength range {y.min():.1f}-{y.max():.1f}')
data.head()

32 runs, 5 factors (2^5 full factorial), strength range 343.2-722.5


,run_order,table_speed_coded,table_speed_m_s,feed_rate_coded,feed_rate_mm,wheel_grit_coded,wheel_grit,direction_coded,direction,batch_coded,batch,strength
0,17,-1.0,0.025,-1.0,0.050,-1.0,140/170,-1.0,longitudinal,-1.0,1,680.45
1,30,1.0,0.125,-1.0,0.050,-1.0,140/170,-1.0,longitudinal,-1.0,1,722.48
2,14,-1.0,0.025,1.0,0.125,-1.0,140/170,-1.0,longitudinal,-1.0,1,702.14
3,8,1.0,0.125,1.0,0.125,-1.0,140/170,-1.0,longitudinal,-1.0,1,666.93
4,32,-1.0,0.025,-1.0,0.050,1.0,80/100,-1.0,longitudinal,-1.0,1,703.67


# Problem 1 (25 pts): map and critique this design

Lecture 38 mapped the 3D-printing dataset and found real flaws (a confounded pair, no true
replication, no run-order column). This dataset was chosen specifically because it does the job
right -- your task is to verify that, not just take it on faith.

(a) Confirm this is a genuine complete factorial: print `x.shape[0]` (rows) next to `2**5`, and
print the pairwise correlation matrix of the 5 coded factors (`x.corr()`). In one sentence, how
does the off-diagonal of this matrix compare to what Lecture 38 found for 3D-printing's
`bed_temperature`/`fan_speed` pair (r = 1.0)?

(b) Randomization check: compute `data['strength'].corr(data['run_order'])`. Is there evidence of
a time-drift confound (a machine warming up, a process degrading run over run)? What would a
correlation near +1 or -1 have told you instead?

(c) Replication check: group by all 5 coded factor columns and report the largest group size
(`data.groupby([...]).size().max()`). Given your answer, can this dataset separate "pure
measurement noise at one condition" from "a real difference between two different conditions"?
Explain in 1-2 sentences.

(d) In 2-3 sentences: this is a well-designed experiment by most of Lecture 38's criteria, but (c)
found one real gap. Name the one thing you'd add to close it, and, using Lecture 38's
(levels)^(factors) cost idea, say roughly how many *additional* runs a single full replicate of
this design would cost.

# Problem 2 (35 pts): analyze the factorial results

Continue using `data`/`x`/`y` from Problem 1.

(a) Compute all 5 main effects on `strength` (mean at coded level +1 minus mean at coded level -1
for each factor), and print them sorted by magnitude, largest first. Which two factors have the
largest-magnitude effects?

(b) Run `scipy.stats.ttest_ind` comparing `strength` where `direction_coded == 1` against where
`direction_coded == -1`. Report the p-value and write one sentence a grinding-shop manager could
act on.

(c) Run the same test for `batch_coded`. Report its p-value. Now suppose you had tested all 5 main
effects and were deciding which to report as "real": compute the Bonferroni-corrected threshold
(0.05 divided by the number of tests) and compare both `direction_coded`'s and `batch_coded`'s
p-values against it. Does the correction change your conclusion about either factor? In 2-3
sentences, what does the answer tell you about how "safe" the `direction_coded` result is, versus
how solid the `batch_coded` result ever was?

(d) Compute the `direction_coded` x `batch_coded` interaction the way Lecture 39 computed
`direction` x `wheel_grit`: build the product column (`data['direction_coded'] *
data['batch_coded']`), split `strength` by whether that product is +1 or -1, and run a t-test.
Report the effect size and p-value. Is there evidence these two factors interact, or do their
effects on strength look additive?

# Problem 3 (40 pts): design your own small factorial experiment

No dataset for this problem -- you are designing the experiment that *would* produce one. This is
a judgment problem: there is no single correct numeric answer, but your choices need to be
justified using today's and last Wednesday's vocabulary.

**Scenario:** A colleague in a thermal-spray coatings lab wants to know whether **nozzle standoff
distance** and **substrate preheat temperature** affect the adhesion strength of a plasma-sprayed
ceramic coating -- and whether the two factors interact. No data exists yet.

Write your answers as markdown (no code required, though you may include a small table or
calculation if it helps).

(a) Choose 2 or 3 factors (you may add a third if you think it matters) and 2 or 3 levels each,
with realistic units. State whether you are running a full factorial or a deliberate compromise,
and if it's a compromise, say what you're giving up to afford it.

(b) State your total run count: (levels)^(factors) times your chosen number of replicates.
Justify the replicate count using Lecture 38's simulation logic -- reason from the n=25/50/100
detection-rate numbers from that lecture, given an adhesion-strength difference you consider
practically meaningful (state that number and why you picked it).

(c) State your randomization plan in one sentence: what run order will you use, and what specific
uncontrolled variable are you protecting against?

(d) Name one thing your own design will **not** be able to tell you, and why -- the same critique
skill from Problem 1(c)-(d), turned on your own design instead of someone else's.

*your Problem 3 answer here*

# Wrap-up

This homework closed the loop Lecture 38 opened: Problem 1 turned the design-critique checklist
on a dataset built to pass it, Problem 2 used that clean design to get an honest answer (one real
effect, one non-effect, no interaction) and applied the multiple-comparisons caution to it, and
Problem 3 asked you to be the one making the design choices instead of auditing someone else's.
Every dataset you touch from here on, in this course or after it, was produced by decisions like
the ones you just made.